# Skill pipeline

Builds the global skill bank for skill-conditioned GRPO, then runs the pilot that decides whether RL is worth running. The untrained model writes the skills once, before any training, and they stay fixed afterward.

At 60 steps, rollouts cover 480 problems and take roughly 3 hours on a T4, and the pilot takes roughly 2 more. `MAX_STEPS` must match the GRPO runs these skills are compared against.

Setup, needed once per account:

1. Select a GPU in **Runtime > Change runtime type**. An A100 or L4 is much faster than a T4.
2. Add a read-only GitHub token for this repository in Colab's **Secrets** panel (the key icon) as `GH_TOKEN`, and turn on notebook access.

Then choose **Runtime > Run all** and check every box on the Google Drive permission popup. Keep this tab open while it runs. If the session disconnects, choose **Run all** again and it continues from where it stopped.

In [ ]:
# Settings. Choose these before the first run.
USE_GOOGLE_DRIVE = True  # False keeps results on the runtime's disk, erased when the session ends.
RUN_NAME = "skills-seed230"
MAX_STEPS = 60  # Must match the GRPO runs these skills are compared against.
ROLLOUT_BATCH_SIZE = 4  # Problems per batch. Fits a T4's memory. Use 8 on an A100 or L4.
PILOT_PROBLEMS = 200  # Problems after the ones GRPO trains on. Must be a multiple of 8.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU found. Select one in Runtime > Change runtime type, then run all.")
get_ipython().system("nvidia-smi --query-gpu=name,memory.total --format=csv")

if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/self-derived-skills/runs")
else:
    RUN_ROOT = Path("/content/runs")
RUN_DIR = RUN_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved to {RUN_DIR}")

REPO_URL = "https://github.com/Ruichu-Xia/self-derived-skills.git"
REPO_DIR = "/content/self-derived-skills"

# The token stays in an environment variable so that it never appears in a command, URL, or error.
GIT_ENV = {**os.environ, "GH_TOKEN": userdata.get("GH_TOKEN")}
CREDENTIAL_HELPER = (
    "credential.helper=!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
)


def git(*args, cwd=None):
    """Run a git command that authenticates with the token in the environment."""
    subprocess.run(["git", "-c", CREDENTIAL_HELPER, *args], env=GIT_ENV, cwd=cwd, check=True)


if os.path.isdir(REPO_DIR):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
get_ipython().system("git log -1 --format='%h %s'")

print("Installing packages. This takes a few minutes.")
install = subprocess.run(["bash", "scripts/colab_install.sh"], capture_output=True, text=True)
print(install.stdout[-1500:])
if install.returncode != 0:
    print(install.stderr[-4000:])
    raise RuntimeError("Installing the packages failed. See the output above.")

## Step 1: collect rollouts

In [ ]:
rollouts_dir = RUN_DIR / "rollouts"
get_ipython().system(
    "python experiments/collect_rollouts.py --config configs/grpo_math.yaml "
    f'--output-dir "{rollouts_dir}" --max-steps {MAX_STEPS} --batch-size {ROLLOUT_BATCH_SIZE}'
)
summary_path = rollouts_dir / "summary.json"
if not summary_path.exists():
    raise RuntimeError("Collecting rollouts did not finish. Choose Run all to continue.")
print(json.dumps(json.loads(summary_path.read_text()), indent=2, sort_keys=True))

## Steps 2 to 4: write, filter, and select skills

In [ ]:
bank_dir = RUN_DIR / "bank"
get_ipython().system(
    "python experiments/build_skill_bank.py --config configs/grpo_math.yaml "
    f'--rollouts "{rollouts_dir / "rollouts.jsonl"}" --output-dir "{bank_dir}"'
)
bank_path = bank_dir / "global_bank.json"
if not bank_path.exists():
    raise RuntimeError("Building the skill bank failed. See the traceback above.")
print(bank_path.read_text())

## Step 5: pilot

In [ ]:
pilot_dir = RUN_DIR / "pilot"
pilot_steps = MAX_STEPS + PILOT_PROBLEMS // 8
for name, bank_flag in [("no-skills", ""), ("with-skills", f'--skill-bank "{bank_path}"')]:
    get_ipython().system(
        "python experiments/collect_rollouts.py --config configs/grpo_math.yaml "
        f'--output-dir "{pilot_dir / name}" --max-steps {pilot_steps} --skip {MAX_STEPS * 8} '
        f"--batch-size {ROLLOUT_BATCH_SIZE} {bank_flag}"
    )
    if not (pilot_dir / name / "summary.json").exists():
        raise RuntimeError(f"The {name} pilot did not finish. Choose Run all to continue.")

get_ipython().system(
    "python experiments/check_skill_pilot.py "
    f'--no-skills "{pilot_dir / "no-skills"}" --with-skills "{pilot_dir / "with-skills"}" '
    f'--output "{RUN_DIR / "pilot.json"}"'
)
pilot = json.loads((RUN_DIR / "pilot.json").read_text())
print("The pilot passed." if pilot["passed"] else "The pilot did not pass. Do not run RL yet.")